In [ ]:
import torch,diffusers

prompts,prompt_2,negative_prompt=["Falling star","Fancy topiary","Grandslam Homerun","Dragon Fire","Boxer versus Ninja"],"natural clarity, sharp focus, intricate detail, expressive, rich deep aesthetic, epic composition","cartoon,bad anatomy,blur"
dtype,model_id=torch.bfloat16,"stabilityai/stable-diffusion-3.5-medium"
prompt_embeddings,pipe=[],diffusers.DiffusionPipeline.from_pretrained(model_id,transformer=None,vae=None,torch_dtype=dtype).to("cuda")
for prompt in prompts:
    with torch.no_grad(): (prompt_embeds,negative_prompt_embeds,pooled_prompt_embeds,negative_pooled_prompt_embeds)=pipe.encode_prompt(prompt=prompt,prompt_2=prompt+prompt_2,prompt_3=prompt_2+prompt,negative_prompt=negative_prompt)
    prompt_embeddings.append((prompt_embeds, negative_prompt_embeds, pooled_prompt_embeds, negative_pooled_prompt_embeds))
del pipe;torch.cuda.empty_cache()
torch.backends.cudnn.benchmark=torch.backends.cuda.matmul.allow_tf32=torch.backends.cudnn.allow_tf32=torch._inductor.config.conv_1x1_as_mm=torch._inductor.config.use_mixed_mm=torch._inductor.config.coordinate_descent_check_all_directions=torch._inductor.config.epilogue_fusion=torch._inductor.config.pattern_matcher=True
torch._inductor.config.triton.cudagraphs,torch._inductor.config.compile_threads,torch.set_float32_matmul_precision= False, 1, "high"
pipe = diffusers.DiffusionPipeline.from_pretrained(model_id,text_encoder=None,text_encoder_2=None,text_encoder_3=None,torch_dtype=dtype).to("cuda")
pipe.transformer.to(memory_format=torch.channels_last)
pipe.enable_xformers_memory_efficient_attention()
pipe.transformer=torch.compile(pipe.transformer,backend="inductor",mode="default",dynamic=False,fullgraph=False)
with torch.inference_mode():
    for i, (prompt, (prompt_embeds, negative_prompt_embeds, pooled_prompt_embeds, negative_pooled_prompt_embeds)) in enumerate(zip(prompts, prompt_embeddings)):
        image=pipe(prompt_embeds=prompt_embeds,negative_prompt_embeds=negative_prompt_embeds,pooled_prompt_embeds=pooled_prompt_embeds,negative_pooled_prompt_embeds=negative_pooled_prompt_embeds).images[0]
        display(image)